<a href="https://colab.research.google.com/github/Maryam-Shile/Intuitive_pythonista_projects/blob/main/Bank_account_predicition_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.ensemble import VotingClassifier

from sklearn.metrics import f1_score, accuracy_score, mean_absolute_error
import warnings
warnings.filterwarnings('ignore')

In [3]:
!unzip /content/financial-inclusion-in-africa20250311-22142-nbnoiv.zip -d /content/financial_inclusion

Archive:  /content/financial-inclusion-in-africa20250311-22142-nbnoiv.zip
  inflating: /content/financial_inclusion/manifest-e268f67161c155de502276443b494f7c20250311-22142-1qdfioo.json  
  inflating: /content/financial_inclusion/Train.csv  
  inflating: /content/financial_inclusion/Test.csv  
  inflating: /content/financial_inclusion/VariableDefinitions.csv  
  inflating: /content/financial_inclusion/SampleSubmission.csv  
  inflating: /content/financial_inclusion/StarterNotebook.ipynb  


In [4]:
train = pd.read_csv('/content/financial_inclusion/Train.csv')
test = pd.read_csv('/content/financial_inclusion/Test.csv')

In [5]:
train['uniqueid'] = train['uniqueid'] + ' x ' + train['country']
test['uniqueid'] = test['uniqueid'] + ' x ' + test['country']

In [6]:
X = train.drop(columns = ['uniqueid', 'bank_account'])
y = train['bank_account']

# X['year'] = X['year'].astype(str)
y = y.map({'Yes': 1, 'No': 0})

zindi_test = test.drop(columns = ['uniqueid'])
# zindi_test['year'] = zindi_test['year'].astype(str)


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23524 entries, 0 to 23523
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   country                 23524 non-null  object
 1   year                    23524 non-null  int64 
 2   location_type           23524 non-null  object
 3   cellphone_access        23524 non-null  object
 4   household_size          23524 non-null  int64 
 5   age_of_respondent       23524 non-null  int64 
 6   gender_of_respondent    23524 non-null  object
 7   relationship_with_head  23524 non-null  object
 8   marital_status          23524 non-null  object
 9   education_level         23524 non-null  object
 10  job_type                23524 non-null  object
dtypes: int64(3), object(8)
memory usage: 2.0+ MB


In [7]:
X_train

,country,year,location_type,cellphone_access,household_size,age_of_respondent,gender_of_respondent,relationship_with_head,marital_status,education_level,job_type
12033,Rwanda,2016,Urban,Yes,7,18,Female,Other non-relatives,Single/Never Married,Primary education,Informally employed
11888,Rwanda,2016,Rural,Yes,3,49,Female,Spouse,Married/Living together,Secondary education,Informally employed
20909,Tanzania,2017,Urban,Yes,2,41,Male,Head of Household,Single/Never Married,Primary education,Informally employed
22785,Uganda,2018,Rural,No,5,25,Female,Spouse,Married/Living together,Primary education,Other Income
14323,Rwanda,2016,Rural,Yes,7,46,Male,Head of Household,Married/Living together,Primary education,Informally employed
...,...,...,...,...,...,...,...,...,...,...,...
11964,Rwanda,2016,Rural,Yes,4,27,Female,Spouse,Married/Living together,Primary education,Farming and Fishing
21575,Uganda,2018,Rural,Yes,9,40,Female,Spouse,Married/Living together,Primary education,Self employed
5390,Kenya,2018,Rural,Yes,4,35,Female,Head of Household,Widowed,Tertiary education,Formally employed Government
860,Kenya,2018,Urban,Yes,2,42,Male,Head of Household,Married/Living together,Tertiary education,Self employed


In [35]:
cat_col = ['country', 'location_type',
       'cellphone_access', 'gender_of_respondent', 'relationship_with_head', 'marital_status',
       'education_level', 'job_type']
num_col = ['household_size', 'age_of_respondent']

preprocessor = ColumnTransformer(transformers = [
    ('cat', OneHotEncoder(drop = 'first', sparse_output = False), cat_col),
    ('num', StandardScaler(), num_col)
], remainder = 'passthrough')

grid = GridSearchCV(cv=5, estimator=GradientBoostingClassifier(), n_jobs=-1,
             param_grid={'max_depth': range(3, 10, 2),
                         'n_estimators': range(5, 100, 5)},
             scoring='accuracy', verbose=1)
pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('grid', grid)
])

pipeline.fit(X_train, y_train)
gboost = pipeline.named_steps['grid'].best_estimator_

lgbm = LGBMClassifier(is_unbalance = True, learning_rate = 0.01, min_data_in_leaf = 50, max_depth = 16, num_leaves = 87, random_state = 42, )


pipeline = Pipeline(steps = [
    ('preprocessor', preprocessor),
    ('voting_classifier', VotingClassifier(estimators = [('lgbm', lgbm), ('gboost', gboost)], voting = 'soft', weights = [1, 1]))
])



pipeline.fit(X_train, y_train)
y_pred_train = pipeline.predict(X_test)

print(f'mae: {mean_absolute_error(y_test, y_pred_train)}')



[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 2670, number of negative: 16149
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.006319 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 160
[LightGBM] [Info] Number of data points in the train set: 18819, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.141878 -> initscore=-1.799780
[LightGBM] [Info] Start training from score -1.799780
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
mae: 0.10669500531349628


In [36]:
pipeline.fit(X, y)

y_pred = pipeline.predict(zindi_test)

[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Info] Number of positive: 3312, number of negative: 20212
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.001981 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 161
[LightGBM] [Info] Number of data points in the train set: 23524, number of used features: 30
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.140792 -> initscore=-1.808724
[LightGBM] [Info] Start training from score -1.808724
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50


In [37]:
print(f'zindi test set: {len(zindi_test)}')
print(f'y_pred: {len(y_pred)}')

zindi test set: 10086
y_pred: 10086


In [ ]:
zindi_test.columns

Index(['country', 'year', 'location_type', 'cellphone_access',
       'household_size', 'gender_of_respondent', 'relationship_with_head',
       'marital_status', 'education_level', 'job_type', 'age_group'],
      dtype='object')

In [38]:
final_submission = pd.DataFrame({'unique_id': test['uniqueid'],
                                'bank_account': y_pred })
final_submission

,unique_id,bank_account
0,uniqueid_6056 x Kenya,1
1,uniqueid_6060 x Kenya,1
2,uniqueid_6065 x Kenya,0
3,uniqueid_6072 x Kenya,0
4,uniqueid_6073 x Kenya,0
...,...,...
10081,uniqueid_2998 x Uganda,0
10082,uniqueid_2999 x Uganda,0
10083,uniqueid_3000 x Uganda,0
10084,uniqueid_3001 x Uganda,0


In [39]:
final_submission.to_csv('seventh_submission.csv', index = False)